# 1. 라이브러리 임포트 및 유틸리티 함수 정의

In [1]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import glob

# 사용자 정의 Dataset 클래스
class PostureDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.data = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(image_path).convert("RGB")
        label = row["class_id"]
        if self.transform:
            image = self.transform(image)
        return image, label

# EarlyStopping 클래스
class EarlyStopping:
    def __init__(self, patience=30, delta=0.0, checkpoint_path='checkpoint.pt'):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.checkpoint_path = checkpoint_path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"🟡 EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                print("🛑 EarlyStopping triggered! Stopping training.")
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        """Validation loss가 개선될 때만 모델 저장"""
        torch.save(model.state_dict(), self.checkpoint_path)
        print(f"✅ Model saved to {self.checkpoint_path}")

# 백본 모델 로드 및 분류기 제거 함수
def get_backbone(name):
    if name == 'efficientnet_b0':
        model = models.efficientnet_b0(pretrained=True)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Identity()
    elif name == 'resnet50':
        model = models.resnet50(pretrained=True)
        in_features = model.fc.in_features
        model.fc = nn.Identity()
    elif name == 'mobilenet_v3_large':
        model = models.mobilenet_v3_large(pretrained=True)
        in_features = model.classifier[0].in_features
        model.classifier = nn.Identity()
    elif name == 'convnext_tiny':
        model = models.convnext_tiny(pretrained=True)
        in_features = model.classifier[2].in_features
        model.classifier = nn.Identity()
    else:
        raise ValueError(f"Unknown model name: {name}")
    
    return model, in_features

# MLP Head 정의 함수
def get_mlp_head(in_features):
    return nn.Sequential(
        nn.BatchNorm1d(in_features),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 1)
    )

# 에폭별 특성 데이터 저장 함수 (날짜 폴더 포함)
def save_epoch_data(epoch, features, labels, setname, backbone, save_dir='embeddings'):
    today = datetime.now().strftime('%Y%m%d')
    save_path = os.path.join(save_dir, today, backbone)
    os.makedirs(save_path, exist_ok=True)
    np.save(f"{save_path}/epoch_{epoch:03d}_{setname}_features.npy", features)
    np.save(f"{save_path}/epoch_{epoch:03d}_{setname}_labels.npy", labels)

# 최신 날짜 폴더를 찾는 헬퍼 함수
def get_latest_date_folder(base_dir):
    date_folders = sorted([f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and f.isdigit() and len(f) == 8], reverse=True)
    if not date_folders:
        raise FileNotFoundError(f"No date-stamped folders found in {base_dir}")
    return date_folders[0]

# T-SNE 시각화 배치 처리 함수
def batch_visualize_tsne(backbone, embedding_dir='embeddings', save_dir='logs'):
    try:
        latest_date_folder = get_latest_date_folder(embedding_dir)
    except FileNotFoundError as e:
        print(f"Error: {e}")
        return

    path = os.path.join(embedding_dir, latest_date_folder, backbone)
    if not os.path.exists(path):
        print(f"No embeddings found for {backbone} in {path}")
        return

    files = sorted([f for f in os.listdir(path) if f.endswith("_features.npy")])
    if not files:
        print(f"No feature files found for {backbone} in {path}")
        return

    for file in files:
        epoch = int(file.split('_')[1])
        features = np.load(os.path.join(path, file))
        labels = np.load(os.path.join(path, file.replace("features.npy", "labels.npy")))

        reduced = PCA(n_components=min(50, features.shape[1]), random_state=42).fit_transform(features)
        embedded = TSNE(n_components=2, random_state=42, perplexity=min(30, len(reduced) - 1)).fit_transform(reduced)

        plt.figure(figsize=(8, 8))
        scatter = plt.scatter(embedded[:, 0], embedded[:, 1], c=labels, cmap='tab10', alpha=0.7)
        plt.colorbar(scatter)
        plt.title(f"{backbone} - Epoch {epoch:03d}")

        out_dir = os.path.join(save_dir, latest_date_folder, backbone)
        os.makedirs(out_dir, exist_ok=True)
        # JPG 포맷으로 저장
        plt.savefig(f"{out_dir}/epoch_{epoch:03d}.jpg")
        plt.close()

# 훈련/검증 데이터 T-SNE 시각화 함수
def visualize_tsne_train_val(epoch, backbone, embedding_dir='embeddings', save_dir='logs'):
    try:
        latest_date_folder = get_latest_date_folder(embedding_dir)
    except FileNotFoundError as e:
        print(f"Error: {e}")
        return

    def load_and_embed(path_feat, path_label):
        features = np.load(path_feat)
        labels = np.load(path_label)
        reduced = PCA(n_components=min(50, features.shape[1]), random_state=42).fit_transform(features)
        embedded = TSNE(n_components=2, random_state=42, perplexity=min(30, len(reduced) - 1)).fit_transform(reduced)
        return embedded, labels

    base = os.path.join(embedding_dir, latest_date_folder, backbone)
    feat_train = os.path.join(base, f"epoch_{epoch:03d}_train_features.npy")
    lbls_train = os.path.join(base, f"epoch_{epoch:03d}_train_labels.npy")
    feat_val = os.path.join(base, f"epoch_{epoch:03d}_val_features.npy")
    lbls_val = os.path.join(base, f"epoch_{epoch:03d}_val_labels.npy")

    if not (os.path.exists(feat_train) and os.path.exists(feat_val)):
        print(f"Features for epoch {epoch:03d} not found for {backbone} in {base}. Skipping visualization.")
        return

    emb_train, y_train = load_and_embed(feat_train, lbls_train)
    emb_val, y_val = load_and_embed(feat_val, lbls_val)

    fig, axs = plt.subplots(1, 2, figsize=(16, 8))
    axs[0].scatter(emb_train[:, 0], emb_train[:, 1], c=y_train, cmap='tab10', alpha=0.7)
    axs[0].set_title(f"Train - Epoch {epoch:03d}")

    axs[1].scatter(emb_val[:, 0], emb_val[:, 1], c=y_val, cmap='tab10', alpha=0.7)
    axs[1].set_title(f"Validation - Epoch {epoch:03d}")

    out_dir = os.path.join(save_dir, latest_date_folder, backbone)
    os.makedirs(out_dir, exist_ok=True)
    # JPG 포맷으로 저장
    plt.savefig(f"{out_dir}/epoch_{epoch:03d}_trainval.jpg")
    plt.close()

# T-SNE 훈련/검증 시각화 일괄 처리 함수
def batch_visualize_tsne_trainval(backbone, embedding_dir='embeddings', save_dir='logs'):
    try:
        latest_date_folder = get_latest_date_folder(embedding_dir)
    except FileNotFoundError as e:
        print(f"Error: {e}")
        return

    path = os.path.join(embedding_dir, latest_date_folder, backbone)
    if not os.path.exists(path):
        print(f"No embeddings found for {backbone} in {path}")
        return

    files = sorted([f for f in os.listdir(path) if f.endswith("_train_features.npy")])
    if not files:
        print(f"No train feature files found for {backbone} in {path}")
        return

    epochs = [int(f.split('_')[1]) for f in files]

    for epoch in epochs:
        visualize_tsne_train_val(epoch, backbone, embedding_dir, save_dir)

# 학습 에폭 함수
def train_epoch(loader, model, criterion, optimizer, device, is_train=True):
    model.train() if is_train else model.eval()
    running_loss = 0.0
    preds, labels = [], []

    for images, targets in loader:
        images = images.to(device)
        targets = targets.float().to(device).unsqueeze(1)

        if is_train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        if is_train:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds_batch = (torch.sigmoid(outputs).detach().cpu().numpy() > 0.5).astype(int)
        targets_batch = targets.detach().cpu().numpy().astype(int)

        preds.extend(preds_batch)
        labels.extend(targets_batch)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    return running_loss / len(loader.dataset), acc, f1

In [3]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


# 2. 데이터 준비 및 전처리

In [ ]:
# CSV 파일 불러오기
train_df = pd.read_csv("../dataset-modification/train_pose_parsed.csv")[["filename", "class_id"]]
valid_df = pd.read_csv("../dataset-modification/valid_pose_parsed.csv")[["filename", "class_id"]]

# filename 중복 처리 (class_id의 min 값 선택)
train_df = train_df.groupby("filename")["class_id"].min().reset_index()
valid_df = valid_df.groupby("filename")["class_id"].min().reset_index()

# 클래스 가중치 계산
class_weights = compute_class_weight('balanced', classes=np.unique(train_df["class_id"]), y=train_df["class_id"])
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# 이미지 경로 정의
train_image_dir = "../dataset-modification/train-visualized/images/"
valid_image_dir = "../dataset-modification/valid-visualized/images/"

# 데이터 증강 및 정규화 Transform 정의
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset 생성
train_dataset = PostureDataset(train_df, train_image_dir, transform=train_transform)
val_dataset = PostureDataset(valid_df, valid_image_dir, transform=val_transform)

# DataLoader 생성 (배치 크기 32)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=os.cpu_count() // 2 or 1) # num_workers 추가
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=os.cpu_count() // 2 or 1) # num_workers 추가

print("Train 클래스 분포:\n", train_df["class_id"].value_counts())
print("Valid 클래스 분포:\n", valid_df["class_id"].value_counts())
print("Class weights:", class_weights_tensor)

NameError: name 'pd' is not defined

In [4]:
original_train_df = pd.read_csv("../dataset-modification/train_pose_parsed.csv")
print("원본 row 수:", len(original_train_df))
print("고유 filename 수:", original_train_df["filename"].nunique())


원본 row 수: 2911
고유 filename 수: 2581


# 3. 다양한 백본 모델 학습 및 특성 추출

In [5]:
# 장치 설정
device = torch.device("mps" if torch.backends.mps.is_available() and torch.backends.mps.is_built()
                      else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 백본 모델 선택지 및 출력 feature dimension 정의
backbone_options = {
    "resnet50": models.resnet50,
    "mobilenet_v3_large": models.mobilenet_v3_large,
    "efficientnet_b0": models.efficientnet_b0,
    "convnext_tiny": models.convnext_tiny # ConvNeXT 추가
}

backbone_feature_dims = {
    "resnet50": 2048,
    "mobilenet_v3_large": 960,
    "efficientnet_b0": 1280,
    "convnext_tiny": 768 # convnext_tiny의 기본 feature dim
}

num_epochs = 40
log_interval = 1 # 모든 에폭마다 특성 저장

# 여러 백본 모델에 대해 반복 실험
for backbone_name, _ in backbone_options.items(): # _는 함수 자체를 사용하는 대신 get_backbone으로 통일
    print(f"\n--- 🔍 Training with Backbone: {backbone_name} ---")

    # 백본 모델 로드 및 MLP 헤드 구성
    backbone, in_features = get_backbone(backbone_name)
    mlp_head = get_mlp_head(in_features) # 드롭아웃 비율 통일
    model = nn.Sequential(backbone, mlp_head)
    model.to(device)

    # 손실 함수 및 최적화기 설정 (클래스 가중치 적용)
    # BCEWithLogitsLoss의 pos_weight는 양성 클래스(class_id=1)에 대한 가중치이므로 class_weights_tensor[1] 사용
    criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights_tensor[1].to(device)) 
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # TensorBoard Writer 설정
    writer = SummaryWriter(log_dir=f"runs/{backbone_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}")

    # 학습 루프
    for epoch in range(num_epochs):
        train_loss, train_acc, train_f1 = train_epoch(train_loader, model, criterion, optimizer, device, is_train=True)
        val_loss, val_acc, val_f1 = train_epoch(val_loader, model, criterion, optimizer, device, is_train=False)

        print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Train F1={train_f1:.4f} | "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}, Val F1={val_f1:.4f}")
        
        # TensorBoard에 손실 기록
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/validation', val_loss, epoch)
        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/validation', val_acc, epoch)
        writer.add_scalar('F1_Score/train', train_f1, epoch)
        writer.add_scalar('F1_Score/validation', val_f1, epoch)

        # 에폭마다 특성 추출 및 저장
        if epoch % log_interval == 0:
            model.eval() # 특징 추출 시 모델 평가 모드
            
            # 훈련 데이터 특성 추출
            features_list, labels_list = [], []
            with torch.no_grad():
                for imgs, lbls in train_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    feats = backbone(imgs) # 백본으로부터 특징 추출
                    if backbone_name.startswith("convnext"): # ConvNeXT는 마지막이 Classifier이므로 GAP 적용
                        feats = F.adaptive_avg_pool2d(feats, (1, 1)).view(feats.size(0), -1)
                    features_list.append(feats.cpu())
                    labels_list.append(lbls.cpu())
            all_train_feats = torch.cat(features_list, dim=0).numpy()
            all_train_lbls = torch.cat(labels_list, dim=0).numpy()
            save_epoch_data(epoch, all_train_feats, all_train_lbls, "train", backbone_name)

            # 검증 데이터 특성 추출
            features_list, labels_list = [], []
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    feats = backbone(imgs) # 백본으로부터 특징 추출
                    if backbone_name.startswith("convnext"):
                        feats = F.adaptive_avg_pool2d(feats, (1, 1)).view(feats.size(0), -1)
                    features_list.append(feats.cpu())
                    labels_list.append(lbls.cpu())
            all_val_feats = torch.cat(features_list, dim=0).numpy()
            all_val_lbls = torch.cat(labels_list, dim=0).numpy()
            save_epoch_data(epoch, all_val_feats, all_val_lbls, "val", backbone_name)

    writer.close()
    print(f"--- ✅ Finished: {backbone_name} Training ---")

Using device: mps

--- 🔍 Training with Backbone: resnet50 ---


/opt/miniconda3/envs/Pose-Detection/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/miniconda3/envs/Pose-Detection/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/miniconda3/envs/Pose-Detection/lib/python3.8/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/miniconda3/envs/Pose-Detection/lib/python3.8/multiprocessing/spawn.py", line 126, in _main


RuntimeError: DataLoader worker (pid(s) 9120) exited unexpectedly

# 4. 특징 임베딩 시각화 (JPG 저장)

In [ ]:
# 각 백본에 대해 T-SNE 시각화 (단일)
for backbone_name in backbone_options.keys():
    print(f"\n--- ✨ Visualizing TSNE for: {backbone_name} ---")
    batch_visualize_tsne(backbone_name)
    print(f"--- ✅ Finished TSNE visualization for {backbone_name} ---")

# 각 백본에 대해 T-SNE 시각화 (훈련/검증 분리)
for backbone_name in backbone_options.keys():
    print(f"\n--- ✨ Visualizing Train/Val TSNE for: {backbone_name} ---")
    batch_visualize_tsne_trainval(backbone_name)
    print(f"--- ✅ Finished Train/Val TSNE visualization for {backbone_name} ---")